# Design and implement data pipelines with Azure Databricks

> **Industry context — StayWell Hotels**
>
> StayWell Hotels operates 50+ properties across Europe and the Middle East. Each night,
> property-management systems (PMS) export reservation events to cloud storage, and a loyalty
> CRM system produces updated guest profiles. The data engineering team must build a
> **medallion pipeline** that ingests this raw data, cleans and validates it, and delivers
> trusted KPIs to Revenue Management and Guest Experience teams.
>
> All demos in this notebook are grounded in this hospitality scenario.

In [ ]:
from scripts.setup import setup, setup_10

spark.sql("USE CATALOG trainer_demo")
setup_10(spark)

## Design order of operations for a pipeline

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-pipelines.design-order-operations-pipeline.png)

### 🎯 Demo: Explore the raw hospitality data (Bronze — Ingest)

**Scenario:** StayWell Hotels' PMS nightly export has landed in the source table `trainer_demo.demo_10.raw_reservations`.  
Walk the audience through the raw data to explain what the Bronze layer looks like before any cleaning.

**Trainer talking points:**
- This is the entry point of the pipeline — data preserved in original form.
- Point out NULLs, invalid statuses, negative amounts — we'll deal with these in Silver.
- Incremental: each nightly run appends new records; we never overwrite Bronze.

In [ ]:

# ── INGEST STAGE: Peek at raw reservation data ──────────────────────────────
# Trainer: run this cell to show what the Bronze layer looks like.
# Point out NULLs, negative amounts, invalid statuses — these are intentional.

raw = spark.read.table("trainer_demo.demo_10.raw_reservations")
print(f"Total raw reservation records: {raw.count():,}")
raw.show(10, truncate=False)


In [ ]:

# ── Show data quality issues in the raw source ───────────────────────────────
# Trainer: walk through each issue to motivate the need for the Silver layer.

from pyspark.sql import functions as F

raw = spark.read.table("trainer_demo.demo_10.raw_reservations")

print("=== Data quality issues in raw_reservations ===\n")

print(f"NULL reservation_id : {raw.filter(F.col('reservation_id').isNull()).count():>6,}")
print(f"NULL guest_id        : {raw.filter(F.col('guest_id').isNull()).count():>6,}")
print(f"Negative total_amount: {raw.filter(F.col('total_amount') <= 0).count():>6,}")
print(f"Invalid status       : {raw.filter(~F.col('status').isin('confirmed','checked_in','checked_out','cancelled')).count():>6,}")
print(f"check_out < check_in : {raw.filter(F.col('check_out_date') < F.col('check_in_date')).count():>6,}")
print(f"\nTotal records        : {raw.count():>6,}")


## Choose notebook vs Lakeflow Pipelines

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-pipelines.choose-notebook-vs-lakeflow-pipelines.png)

### 🎯 Demo: When to choose notebooks vs Lakeflow Spark Declarative Pipelines

**Scenario:** Walk through a **procedural** notebook example (step-by-step) vs a **declarative** pipeline (file reference). Highlight the key differences for the StayWell Hotels use case.

**Trainer talking points:**
- Notebook approach: full Python control, explicit read → transform → write; good for complex custom logic (e.g. classify_cancellation_reason UDF).
- Declarative approach: define *what* the result should look like; framework handles orchestration, incremental processing and retry.
- Both approaches can coexist — notebooks for custom logic, Lakeflow Pipelines for core ETL.

The cells below show the **notebook / procedural approach**. The declarative approach is demonstrated later using the pipeline files in `pipelines/10-data-pipelines/`.

In [ ]:

# ── PROCEDURAL APPROACH: read → transform → write in a notebook ──────────────
# Trainer: this is the notebook/imperative style.
# Contrast with the declarative @dp.table definitions in the pipeline files.

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# STEP 1: Read (explicit, imperative)
raw_reservations = spark.read.table("trainer_demo.demo_10.raw_reservations")

# STEP 2: Transform — apply business logic in Python
#  Custom logic: classify booking channel into a broader segment.
#  This kind of conditional Python logic is difficult to express declaratively.
def classify_channel(channel):
    if channel in ("direct",):           return "direct"
    if channel in ("ota",):              return "online_intermediary"
    if channel in ("corporate",):        return "b2b"
    if channel in ("travel_agent",):     return "b2b"
    return "other"

classify_udf = F.udf(classify_channel, StringType())

transformed = (
    raw_reservations
    .filter(F.col("reservation_id").isNotNull())
    .filter(F.col("total_amount") > 0)
    .withColumn("channel_segment", classify_udf(F.col("booking_channel")))
    .withColumn(
        "stay_nights",
        F.datediff(F.col("check_out_date"), F.col("check_in_date"))
    )
)

# STEP 3: Write (explicit, imperative)
(
    transformed
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("trainer_demo.demo_10.notebook_reservations_enriched")
)

print(f"Written {transformed.count():,} rows to notebook_reservations_enriched")
print("\nSample output:")
transformed.select("reservation_id", "property_id", "booking_channel",
                   "channel_segment", "stay_nights", "total_amount").show(8)


## Design Lakeflow job logic

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-pipelines.design-task-logic-lakeflow-job.png)

### 🎯 Demo: Lakeflow Job — task logic, DAG dependencies, and task values

**Scenario:** StayWell's nightly pipeline is orchestrated as a Lakeflow Job with four tasks:
1. `ingest_reservations` — runs the bronze streaming ingestion notebook
2. `clean_reservations` — Silver cleaning  
3. `clean_guests` — Silver guests cleaning (runs **in parallel** with task 2)
4. `build_gold_kpis` — depends on both task 2 and 3 (fan-in)

**Trainer talking points:**
- Open the Lakeflow Jobs UI and draw the DAG live, pointing out parallel vs sequential tasks.
- Show the **task dependency conditions** table from the learn unit (All succeeded / None failed / All done).
- The cell below demonstrates `dbutils.jobs.taskValues` — how tasks pass data to downstream tasks without writing to a table.
- Point out that the `for_each` region processes each of the 10 hotel properties independently.

In [ ]:

# ── Demonstrate dbutils.jobs.taskValues ──────────────────────────────────────
# Trainer: explain that in a real Lakeflow Job each task would be a separate
# notebook. Here we simulate setting and getting task values in-notebook for demo.

# Simulate Task 1 (ingest_reservations) setting a task value
# In a real job this code would live in the ingest_reservations notebook.

from pyspark.sql import functions as F

reservations = spark.read.table("trainer_demo.demo_10.raw_reservations")
record_count = reservations.count()

# In a Lakeflow Job this would be:
#   dbutils.jobs.taskValues.set(key="reservations_ingested", value=record_count)
print(f"[Simulated task value] reservations_ingested = {record_count:,}")

# Demonstrate getting property list for a For each task
properties = (
    reservations
    .select("property_id")
    .distinct()
    .orderBy("property_id")
    .rdd.flatMap(lambda r: [r[0]])
    .collect()
)

# In a real job:
#   dbutils.jobs.taskValues.set(key="properties_to_process", value=properties)
print(f"\n[Simulated task value] properties_to_process = {properties}")
print(f"\nA 'For each' task would iterate over these {len(properties)} properties.")
print("Each iteration processes one property independently (configurable concurrency).")


In [ ]:

# ── Demonstrate serial vs parallel execution in the pipeline DAG ─────────────
# Trainer: show that tasks without dependencies run in parallel.
# Compare time for serial processing (one property at a time) vs parallel.

import time
from pyspark.sql import functions as F

raw = spark.read.table("trainer_demo.demo_10.raw_reservations")

# SERIAL: process each property one after another
print("=== Serial processing (sequential, one property at a time) ===")
t0 = time.time()
for prop in properties[:3]:  # show just 3 properties for demo speed
    count = raw.filter(F.col("property_id") == prop).count()
    print(f"  {prop}: {count:,} reservations")
serial_time = time.time() - t0
print(f"  Elapsed (serial, 3 properties): {serial_time:.1f}s\n")

# PARALLEL (conceptual): in a real Lakeflow Job these would run as parallel tasks
# The framework handles scheduling and resource allocation automatically.
print("=== Parallel processing (as Lakeflow For each task with concurrency=3) ===")
print("  In a Lakeflow Job, all 3 properties would run concurrently.")
print("  Elapsed would be ~1/3 of serial time (limited by slowest partition).")
print("\n  Configure concurrency on the 'For each' task in the job UI.")


## Design error handling in pipelines and jobs

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-pipelines.design-error-handling-pipelines.png)

### 🎯 Demo: Error handling in notebooks — try/except, retry, and dbutils.notebook.exit

**Scenario:** StayWell's cancellation-analysis notebook must handle transient failures gracefully and signal its result back to the Lakeflow Job.

**Trainer talking points:**
- Show the three error handling levels: infrastructure (handled by Azure Databricks), data-level, application logic.
- Walk through `try/except` catching `PySparkException` with `getErrorClass()`.
- Show `run_with_retry()` for transient failures (e.g., network timeout reading from a PMS API).
- Show `dbutils.notebook.exit()` — the downstream task uses the exit value in a conditional dependency.
- The declarative pipeline alternative (expectations) is shown in the pipeline files.

In [ ]:

# ── Error handling: try/except with PySparkException ─────────────────────────
# Trainer: demonstrate data-level error handling in a notebook pipeline task.
# This is error handling the data engineer is responsible for (NOT infrastructure).

from pyspark.errors import PySparkException
from pyspark.sql import functions as F

# Simulate handling a missing table gracefully
try:
    # Attempt to read the silver reservations (may not exist if pipeline hasn't run)
    silver = spark.read.table("trainer_demo.demo_10.silver_reservations")
    count = silver.count()
    print(f"✓ silver_reservations has {count:,} records")

except PySparkException as e:
    if e.getErrorClass() == "TABLE_OR_VIEW_NOT_FOUND":
        print(f"⚠ Table not found: {e.getMessageParameters().get('relationName', 'unknown')}")
        print("  Downstream steps will be skipped. Run the Lakeflow Pipeline first.")
    else:
        raise  # Re-raise unexpected errors

except Exception as e:
    print(f"✗ Unexpected error: {e}")
    raise


In [ ]:

# ── Error handling: retry with exponential backoff ───────────────────────────
# Trainer: show how to handle transient failures (e.g., network call to PMS API).
# StayWell's PMS API occasionally returns 503 during peak booking windows.

import time

def run_with_retry(operation, max_retries=3, base_delay=2):
    """Retry an operation with exponential backoff."""
    for attempt in range(max_retries + 1):
        try:
            return operation()
        except Exception as e:
            if attempt == max_retries:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"  Attempt {attempt + 1} failed: {e}. Retrying in {delay}s...")
            time.sleep(delay)

# Example: simulate a flaky read operation
import random

attempt_counter = {"n": 0}

def flaky_reservation_count():
    """Simulate an operation that fails on first 2 attempts."""
    attempt_counter["n"] += 1
    if attempt_counter["n"] < 3:
        raise ConnectionError("PMS API timeout — transient error")
    return spark.read.table("trainer_demo.demo_10.raw_reservations").count()

print("Calling flaky_reservation_count() with retry logic...")
count = run_with_retry(flaky_reservation_count, max_retries=3, base_delay=1)
print(f"\n✓ Successfully retrieved count after {attempt_counter['n']} attempt(s): {count:,} records")


## Create pipeline with notebook

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-pipelines.create-pipeline-notebook-precedence.png)

### 🎯 Demo: Create a pipeline with a notebook task and task dependencies

**Scenario:** Build a Lakeflow Job that runs StayWell's cancellation-analysis notebook as a task.

**Step-by-step instructions for the trainer:**

1. In your Databricks workspace, open **Jobs & Pipelines** → **Create** → **Job**.
2. Name the job `staywell_nightly_pipeline`.
3. Add **Task 1**: type = Notebook, name = `ingest_reservations`, path = `Demo/pipelines/10-data-pipelines/04_notebook_pipeline_task`.
4. Add **Task 2**: type = Notebook, name = `build_cancellation_report`, depends on `ingest_reservations`.
   - Set parameter `process_date` to `{{job.start_time.iso_date}}`.
5. Show the DAG — point out that Task 2 only runs after Task 1 succeeds.
6. Click **Run now** → watch the task graph execute sequentially.

**Key concepts to highlight:** task dependencies, parameter passing via widgets, `dbutils.notebook.exit()` signals, conditional task flows.

The code for Task 1 is in `pipelines/10-data-pipelines/04_notebook_pipeline_task.py`:

In [ ]:

# ── Notebook task: widgets and parameter passing ─────────────────────────────
# Trainer: paste this into the FIRST CELL of the notebook task you create in
# the Lakeflow Job. It shows how parameters flow from the job into the notebook.
#
# When run interactively (in this demo notebook), the widget defaults are used.
# When run as a job task the job can override the values via the task parameters.

dbutils.widgets.text("process_date",  "", "Process Date (YYYY-MM-DD)")
dbutils.widgets.text("property_id",   "ALL", "Property ID (or ALL)")

process_date = dbutils.widgets.get("process_date") or str(spark.sql("SELECT current_date()").collect()[0][0])
property_id  = dbutils.widgets.get("property_id")

print(f"Running cancellation analysis for date: {process_date}, property: {property_id}")


In [ ]:

# ── Notebook task: dbutils.notebook.exit to signal result to job ─────────────
# Trainer: this cell shows how a notebook communicates success/failure back
# to the Lakeflow Job. The exit value can be read by downstream tasks.
#
# The downstream task can use an If/else condition:
#   {{tasks.ingest_reservations.values.cancellations_processed}} > 0
# to decide whether to send a notification or skip reporting.

from pyspark.sql import functions as F

try:
    raw = spark.read.table("trainer_demo.demo_10.raw_reservations")

    cancelled = raw.filter(F.col("status") == "cancelled")
    if property_id != "ALL":
        cancelled = cancelled.filter(F.col("property_id") == property_id)

    records = cancelled.count()

    # Set a task value so downstream tasks can read it
    # dbutils.jobs.taskValues.set(key="cancellations_processed", value=records)
    print(f"✓ Found {records:,} cancelled reservations")
    print(f"  (In a Lakeflow Job: dbutils.notebook.exit('SUCCESS: {records}'))")

except Exception as e:
    print(f"✗ Error: {e}")
    # In a Lakeflow Job: dbutils.notebook.exit(f"FAILED: {str(e)}")
    # The job would mark this task as failed and trigger any configured alerts.


## Create pipeline with Lakeflow Spark Declarative Pipelines

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-pipelines.create-pipeline-lakeflow-declarative.png)

### 🎯 Demo: Create a Lakeflow Spark Declarative Pipeline for StayWell Hotels

**Step-by-step instructions for the trainer:**

1. In your Databricks workspace, open **Jobs & Pipelines** → **Create** → **ETL Pipeline**.
2. Name the pipeline `staywell_hotel_pipeline`.
3. Set the **Source code** to the folder `Demo/pipelines/10-data-pipelines` (or add individual files).
4. Set the **Target schema** to `trainer_demo.demo_10`.
5. Set **Pipeline mode** to **Triggered** (for batch demos) or **Continuous** (for streaming).
6. Click **Save** → **Start** (or **Dry run** to validate without running).

**Walk through the pipeline files in order:**

| File | Layer | What it demonstrates |
|---|---|---|
| `01_bronze_reservations.py` | Bronze | `@dp.table` for streaming ingestion, append-only |
| `02_silver_clean_validate.py` | Silver | Expectations: warn / drop / fail |
| `03_gold_transforms.py` | Gold | `@dp.materialized_view` for aggregations and joins |

**Concepts to highlight after running:**
- The **DAG view** — automatic dependency ordering, no need to write orchestration code.
- The **Data quality tab** — pass/fail counts for each expectation.
- **Incremental processing**: only new Bronze records are processed on each run.
- Contrast `@dp.table` (streaming, append-only) vs `@dp.materialized_view` (re-computable, handles updates).

In [ ]:

# ── Preview the declarative pipeline source files ────────────────────────────
# Trainer: read and display the pipeline source files so the audience can see
# the declarative syntax alongside the running pipeline in the UI.

import os

pipeline_dir = "pipelines/10-data-pipelines"
pipeline_files = sorted([
    f for f in os.listdir(pipeline_dir) if f.endswith(".py")
])

print(f"Pipeline source files in {pipeline_dir}/:\n")
for fname in pipeline_files:
    fpath = os.path.join(pipeline_dir, fname)
    with open(fpath) as fh:
        first_lines = [l.rstrip() for l in fh.readlines()[:8]]
    print(f"{'─'*60}")
    print(f"  📄 {fname}")
    print(f"{'─'*60}")
    for line in first_lines:
        print(f"  {line}")
    print()


In [ ]:

# ── After running the pipeline: query the Gold layer results ─────────────────
# Trainer: run this AFTER the Lakeflow pipeline completes at least one run.
# Show the audience the final Gold tables that the BI tools consume.

from pyspark.sql import functions as F

try:
    # Revenue summary by property
    print("=== gold_property_revenue_summary ===")
    spark.read.table("trainer_demo.demo_10.gold_property_revenue_summary") \
         .orderBy(F.col("monthly_revenue").desc()) \
         .show(10)

except Exception as e:
    print(f"⚠ Gold tables not available yet — run the Lakeflow Pipeline first.\n  Error: {e}")

try:
    # Top 5 properties by daily occupancy
    print("=== gold_daily_occupancy (top 5 days by revenue) ===")
    spark.read.table("trainer_demo.demo_10.gold_daily_occupancy") \
         .orderBy(F.col("total_revenue").desc()) \
         .show(5)

except Exception as e:
    print(f"⚠ gold_daily_occupancy not available — run the Lakeflow Pipeline first.\n  Error: {e}")
